# Roosevelt National Forest of northern Colorado Competition

In this experiment we will apply some classifiers based on trees and then stack them all together to find out a good classifier

Ensemble Methods: RF, AdaBoost

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import math

%matplotlib inline

In [ ]:
# Global parameters
#root folder
root_folder='..\..'
#data_folder='.'
data_folder_name='Datasets/RooseveltForest'

# Variable for data directory
DATA_PATH = os.path.abspath(os.path.join(root_folder, data_folder_name))

# Both train and test set are in the root data directory
train_path = DATA_PATH
test_path = DATA_PATH


## Read training and test data

In [ ]:
# Reading the csv file with the whole dataset
data = pd.read_csv(train_path+'\\train.csv', sep=',', header=0, index_col=0)
# Train data, erasing label column
train_data = data.drop('Cover_Type', axis=1)
train_label = data[['Cover_Type']]
# Read the test data
test_data = pd.read_csv(test_path+'\\test.csv', sep=',', header=0, index_col=0)

## Data preprocessing

### Global variables

In [ ]:
# Define some groups of columns based on  type
var_numeric = ['Elevation','Aspect','Slope','Horizontal_Distance_To_Hydrology','Vertical_Distance_To_Hydrology',
                'Horizontal_Distance_To_Roadways','Hillshade_9am','Hillshade_Noon','Hillshade_3pm',
                'Horizontal_Distance_To_Fire_Points']
# There are no categoric variables to transfomr
var_categoric = []

# Numerical columns with missing values
# In our first approach, no missing values to consider
columns_nan=[]

#Defining some relevant group of columns
# Columns to be ingored for classification
# Soil Type 7,8, 15 y 25
columns_to_ignore = ['Soil_Type7','Soil_Type8','Soil_Type15','Soil_Type25']

num_columns_nan=[i for i in columns_nan if i not in columns_to_ignore]
enconde_cols_name=[i for i in var_categoric if i not in columns_to_ignore]

### Feature engineering and selection

In [ ]:
# Horizontal to roadways and to Fire poins are very simila recpect to arget variable so lets drop horizontal to roadways
# Horizontal and vertical to hidrology are correlated, lets drop vertical
# Drop soil types where the number of rows are very low: 9, 21, 27, 28, 36
# Outliers: 
columns_unselect=['Soil_Type9','Soil_Type21','Soil_Type27','Soil_Type28','Soil_Type36']


In [ ]:
# Manipulate and transform columns in dataframe to be used in the next stages

# Delete non relevant columns
train_data.drop(columns_to_ignore+columns_unselect, axis=1,inplace=True)
test_data.drop(columns_to_ignore+columns_unselect, axis=1,inplace=True)

#Filling na values in columns
#Not neccesary at this moment

#Dealing with ourliers or data errors
#Not neccesary at this moment

# Define columns to fill nan values and categorial columns to one-hot-encode 
columns_nan=[train_df.columns.tolist().index(i) for i in num_columns_nan]
columns_enconde=[train_df.columns.tolist().index(i) for i in enconde_cols_name]


In [ ]:
print('Train data: ',train_data.shape)
print('Train labels: ',train_label.shape)
print('Test data: ',test_data.shape)

## Preparing the data for the algorithm

In [ ]:
# Tranform to numpy array
X = train_data.values.astype('float64')
y = train_label.values.ravel()
test_ds= test_data.values.astype('float64')

#Convertimos todos los valores a float incluida la columna boolean
# To avoid error in the imputer action if it was neccesary
#X= np.vstack(X[:, :]).astype(np.float)

print('X: ', X.shape)
print('y: ', y.shape)
print('test: ', test_ds.shape)
#X Shape 500,000 x 8

In [ ]:
X

## Building and training an algorithm

### Creating the train and test datasets

In [ ]:
# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4, random_state = 0)

# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
test_ds= sc.transform(test_ds)

In [ ]:
print('X Train Shape:', X_train.shape)
print('X Test Shape:', X_test.shape)
print('Test Shape:', test_ds.shape)

In [ ]:
X_train_l2 = pd.DataFrame(columns=['RF'])
X_train_l2.info

### Train the algorithm: Random Forest

In [ ]:
# Fitting Decision Tree to the Training set
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

parameter_candidates = [{'n_estimators': [50, 100, 125]}]

classifier = GridSearchCV(estimator=RandomForestClassifier(criterion = 'entropy', max_depth=None,min_samples_split=2,
                                                           min_samples_leaf=1, max_leaf_nodes=None,random_state = 0),
                          param_grid=parameter_candidates, scoring='accuracy', n_jobs=-1, cv=5, verbose=1, refit=True, 
                          return_train_score=True)

# Train the classifier on data1's feature and target data
classifier.fit(X_train, y_train)

In [ ]:
#Show the results of the hyperparameter tunning
print(classifier.best_params_)
print(classifier.best_estimator_)


In [ ]:
from sklearn.model_selection import StratifiedKFold

y_preds, y_actual=[],[]
for train_index, test_index in StratifiedKFold(n_splits=10, random_state=1234, shuffle=True).split(X, y):
    y_preds.append(classifier.best_estimator_.predict_proba(X[test_index]))
    y_actual.append(y[test_index])



In [ ]:
from sklearn.metrics import accuracy_score

predictions=classifier.best_estimator_.predict(X_test)
print(predictions.shape)
predictions_test = classifier.best_estimator_.predict(test_ds)
print(predictions_test.shape)
print(round(accuracy_score(y_test, predictions),4))

X_train_l2['RF'] = predictions

### Train the algorithm: Decision Tree

In [ ]:
# Fitting Decision Tree to the Training set
from sklearn.tree import DecisionTreeClassifier

classifier = DecisionTreeClassifier(criterion = 'entropy',max_depth=None,min_samples_split=2, min_samples_leaf=1,
                                    max_leaf_nodes=None,max_features='auto', random_state = 0)

classifier.fit(X_train, y_train)
#Show the results of the hyperparameter tunning
predictions=classifier.predict(X_test)
print(predictions.shape)
predictions_test = classifier.predict(test_ds)
print(predictions_test.shape)
print(round(accuracy_score(y_test, predictions),4))

X_train_l2['DT'] = predictions

### Train the algorithm: AdaBoosted Decision Tree

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Define the base estimator for the Adaboost algorithm
base_classifier = DecisionTreeClassifier(criterion ='entropy',max_depth=None, min_samples_split=2,min_samples_leaf=2,
                                                       max_leaf_nodes=None,random_state = 0)

parameter_candidates = [{'n_estimators': [50, 100, 125], 'learning_rate': [0.25, 0.5, 1.]}]

classifier = GridSearchCV(estimator=AdaBoostClassifier(base_estimator=base_classifier,random_state=0),
                          param_grid=parameter_candidates, scoring='accuracy', n_jobs=-1, cv=5, verbose=1, refit=True, 
                          return_train_score=True)

classifier.fit(X_train, y_train)

In [ ]:
#Show the results of the hyperparameter tunning
print(classifier.best_params_)
print(classifier.best_estimator_)

predictions=classifier.best_estimator_.predict(X_test)
print(predictions.shape)
predictions_test = classifier.best_estimator_.predict(test_ds)
print(predictions_test.shape)
print(round(accuracy_score(y_test, predictions),4))

X_train_l2['AB'] = predictions

### Train the algorithm: Gradient Boosting Classifier

In [ ]:
# Fitting Decision Tree to the Training set
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
#from sklearn.tree import DecisionTreeClassifier
#classifier = GradientBoostingClassifier(n_estimators=100,learning_rate=0.5,max_depth=15,
#              min_samples_leaf=1, min_samples_split=2,random_state=0,verbose=1)
#Hyperparameter tuning with Grid Search
parameter_candidates = [{'n_estimators': [50, 100, 125], 'learning_rate': [0.5,0.2,0.1], 'max_depth':[5,10,None]}]

classifier = GridSearchCV(estimator=GradientBoostingClassifier(min_samples_leaf=1, min_samples_split=2,random_state=0,verbose=1),
                          param_grid=parameter_candidates, n_jobs=-1, cv=5, verbose=1, return_train_score=True)

# Train the classifier on data1's feature and target data
classifier.fit(X_train, y_train)


In [ ]:
#Show the results of the hyperparameter tunning
print(classifier.best_params_)
print(classifier.best_estimator_)

predictions=classifier.best_estimator_.predict(X_test)
print(predictions.shape)
predictions_test = classifier.best_estimator_.predict(test_ds)
print(predictions_test.shape)
print(round(accuracy_score(y_test, predictions),4))

X_train_l2['GB'] = predictions

In [ ]:
# Scoring the model by Cross Validation
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
mean_score = accuracies.mean()
std_score = accuracies.std()
print('Mean: ',mean_score)
print('Std: ',std_score)

In [ ]:
# Fitting Decision Tree to the Training set
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV

parameter_candidates = [{'n_estimators': [50, 100, 125]}]

classifier = GridSearchCV(estimator=ExtraTreesClassifier(criterion = 'entropy', max_depth=None,min_samples_split=2,
                                                           min_samples_leaf=1, max_leaf_nodes=None,random_state = 0),
                          param_grid=parameter_candidates, scoring='accuracy', n_jobs=-1, cv=5, verbose=1, refit=True, 
                          return_train_score=True)

# Train the classifier on data1's feature and target data
classifier.fit(X_train, y_train)

In [ ]:
#Show the results of the hyperparameter tunning
print(classifier.best_params_)
print(classifier.best_estimator_)

predictions=classifier.best_estimator_.predict(X_test)
print(predictions.shape)
predictions_test = classifier.best_estimator_.predict(test_ds)
print(predictions_test.shape)
print(round(accuracy_score(y_test, predictions),4))

X_train_l2['ET'] = predictions

In [ ]:
# Fitting Decision Tree to the Training set
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier

parameter_candidates = [{'max_iter': [50, 100, 125], 'learning_rate': [0.5,0.2,0.1]}]

classifier = GridSearchCV(estimator=GradientBoostingClassifier(min_samples_leaf=20, random_state=0,verbose=1),
                          param_grid=parameter_candidates, n_jobs=-1, cv=5, verbose=1, return_train_score=True)

# Train the classifier on data1's feature and target data
classifier.fit(X_train, y_train)

### Scoring and evaluating the model

In [ ]:
# Predicting the Test set results
y_pred = classifier.predict(X_test)

# Making the Confusion Matrix
from sklearn.metrics import confusion_matrix,accuracy_score,roc_auc_score,roc_curve,auc
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix: ',cm)

acc = accuracy_score(y_test, y_pred)
print('Accuracy: ',acc)


## Showing the results

In [ ]:
import seaborn as sn

df_cm = pd.DataFrame(cm, range(cm.shape[0]),
                  range(cm.shape[1]))
#plt.figure(figsize = (10,7))
sn.set(font_scale=1.4) #for label size
sn.heatmap(df_cm, annot=True,fmt='.2f',annot_kws={"size": 10})# font size

In [ ]:
# ROC Curve
# plot no skill
plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()


## Building and training a XGBoosted Classifier

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

### Simple train and score 

In [ ]:
model = XGBClassifier()
model.fit(X_train, y_train)
# make predictions for test data
y_pred = model.predict(X_test)
predictions = [round(value) for value in y_pred]
# evaluate predictions
accuracy = accuracy_score(y_test, predictions)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

### Use early stopping to prevent overfiting

In [ ]:
# fit model on training data
model = XGBClassifier()
eval_set = [(X_test, y_test)]
model.fit(X_train, y_train, early_stopping_rounds=10, eval_metric="mlogloss", eval_set=eval_set, verbose=True)
# make predictions for test data
y_pred = model.predict(X_test)
predictions = [round(value) for value in y_pred]
# evaluate predictions
accuracy = accuracy_score(y_test, predictions)
print("Accuracy: %.2f%%" % (accuracy * 100.0))


In [ ]:
print(model.feature_importances_)

In [ ]:
from xgboost import plot_importance
plt.figure(figsize=(10,20))
plot_importance(model)
plt.show()

### Hyperparameter tunning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

model = XGBClassifier()

n_estimators = [50, 100, 150, 200]
max_depth = [2, 4, 6, 8]
param_grid = dict(max_depth=max_depth, n_estimators=n_estimators)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=7)
grid_search = GridSearchCV(model, param_grid, scoring="neg_log_loss", n_jobs=-1, cv=kfold, verbose=1)
result = grid_search.fit(X_train, y_train)

In [ ]:
print("Best: %f using %s" % (result.best_score_, result.best_params_))
means = result.cv_results_['mean_test_score']
stds = result.cv_results_['std_test_score']
params = result.cv_results_['params']

for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

In [ ]:
import sklearn
print('The scikit-learn version is {}.'.format(sklearn.__version__))

In [ ]:
from lightgbm import LGBMModel,LGBMClassifier